# The Ancestor Game

- Jeu JcJ, tour par tour
- État de départ: une forêt orientée de n noeuds
- Objectif: être le premier à capturer au moins la moitié des noeuds
- Pendant leur tours, les joueurs peuvent faire les actions suivantes:
  - Ajouter un parent
    - Le joueur choisit un noeud sans parent et lui en ajoute un.
    - Aucun cycle ne doit se faire
  - Placer un drapeau
    - Le joueur place un drapeau de sa couleur sur un noeud pas capturé (un noeud peut avoir plusieurs drapeaux, mais un seul de chaque couleur)
  - Capture simple
    - Le joueur capture un noeud avec un drapeau de sa couleur
  - Capture combo
    - Le joueur choisi 2 noeuds avec un drapeau de sa couleur
    - Tous les noeuds pas capturés jusqu'au Least Common Ancestor des 2 noeuds sont capturés par le joueur


Un noeud capturé ne peut pas être recapturé. Capturer un noeud enlève tous les drapeux posés dessus.

## Drapeaux

In [31]:
class Flag:
  def __init__(self, pid):
    self.player_id = pid

## Node en général

In [32]:
class Node:
    def __init__(self, id):
        self.id = id
        self.parent = None

    def set_parent(self, node):
        self.parent = node

    def ancestors(self):
        result = []
        current = self
        while current is not None:
            result.append(current)
            current = current.parent
        return result


## Node du jeu

In [33]:
class GameNode(Node):
    def __init__(self, id):
        super().__init__(id)
        self.flags = []
        self.is_captured = False

    def add_flag(self, flag):
        if not self.is_captured: self.flags.append(flag)

    def get_flags(self):
        return self.flags
    
    def capture(self):
        self.is_captured = True
        self.flags = []

    def has_flag_from_player(self, pid):
        for f in self.flags:
            if f.player_id == pid: return True
        return False

## Player

In [34]:
class Player:

    def __init__(self, pid):
        self.player_id = pid
        self.zones_captured = 0

    def can_capture(self, n1, n2):
        if (n1 == n2 and n1.has_flag_from_player(self.player_id)):
            return True

        if (n1.ancestors()[-1] in n2.ancestors() and n1.has_flag_from_player(self.player_id) and n2.has_flag_from_player(self.player_id)):
            return True
        
        return False

    def capture(self, n1, n2):
        if self.can_capture(n1, n2):
            ancestors_n1 = n1.ancestors()
            ancestors_n2 = n2.ancestors()

            ancestors_n1_set = set(id(node) for node in ancestors_n1)
            lca = next(node for node in ancestors_n2 if id(node) in ancestors_n1_set)

            current = n1
            while current != lca:
                if not current.is_captured:
                    current.capture()
                    self.zones_captured += 1
                current = current.parent

            current = n2
            while current != lca:
                if not current.is_captured:
                    current.capture()
                    self.zones_captured += 1
                current = current.parent

            lca.capture()
            self.zones_captured += 1

    def can_place_flag(self, n):
        return not n.is_captured and not n.has_flag_from_player(self.player_id)

    def place_flag(self, n):
        if (n.has_flag_from_player(self.player_id)): return
        n.add_flag(Flag(self.player_id))

    def can_give_parent(self, n_child, n_parent):
        return n_child.parent is None and not n_child in n_parent.ancestors()

    def give_parent(self, n_child, n_parent):
        n_child.set_parent(n_parent)

In [35]:
import networkx as nx
import matplotlib.pyplot as plt
import random

class AncestorGameDisplayer:
    def __init__(self, n):
        self.n = n
        rng = random.Random(42)

        self.parents = [-1] * n

        self.is_red    = [False] * n
        self.is_blue   = [False] * n

        self.has_blue_F = [False] * n
        self.has_red_F  = [False] * n

    def _get_node_color(self, i):
        if self.is_red[i]:  return "red"
        if self.is_blue[i]: return "blue"
        return "black"

    def _get_f_label(self, i):
        if self.has_blue_F[i] and self.has_red_F[i]: return "both"
        if self.has_blue_F[i]: return "blue"
        if self.has_red_F[i]:  return "red"
        return None

    def show(self):
        G = nx.DiGraph()

        for i in range(self.n):
            G.add_node(i, color=self._get_node_color(i),
                          f_label=self._get_f_label(i))

        for i, p in enumerate(self.parents):
            if p != -1:
                G.add_edge(i, p)

        pos = nx.spring_layout(G, seed=42)
        fig, ax = plt.subplots(figsize=(8, 6))

        nx.draw_networkx_edges(
            G, pos, ax=ax,
            arrows=True,
            arrowstyle="-|>",
            arrowsize=20,
            connectionstyle="arc3,rad=0.2",
            edge_color="gray",
            width=2,
        )
        nx.draw_networkx_nodes(
            G, pos, ax=ax,
            node_color=["white"] * self.n,
            node_size=800,
            edgecolors=[self._get_node_color(i) for i in range(self.n)],
            linewidths=3,
        )

        for i in range(self.n):
            f = self._get_f_label(i)
            if f is not None:
                x, y = pos[i]
                if f == "both":
                    # Two F labels side by side, offset left and right
                    ax.text(x - 0.02, y, "F", fontsize=11, fontweight="bold",
                            color="red", ha="center", va="center")
                    ax.text(x + 0.02, y, "F", fontsize=11, fontweight="bold",
                            color="blue", ha="center", va="center")
                else:
                    ax.text(x, y, "F", fontsize=12, fontweight="bold",
                            color=f, ha="center", va="center")
                    
        for i in range(self.n):
            x, y = pos[i]
            ax.text(
                x + 0.05, y + 0.05, str(i),
                fontsize=9,
                color="gray",
                ha="left",   # align from left
                va="bottom"  # align from bottom (top-right corner of node)
            )

        ax.set_title("The Ancestor Game", fontsize=14)
        ax.axis("off")
        plt.tight_layout()
        fig.patch.set_facecolor("white")
        ax.set_facecolor("white")
        plt.show()
    
    def add_edge(self, child, parent):
        self.parents[child] = parent

    def red_player_capture(self, node):
        self.is_red[node] = True
        self.is_blue[node] = False
        self.has_red_F[node] = False
        self.has_blue_F[node] = False

    def blue_player_capture(self, node):
        self.is_red[node] = False
        self.is_blue[node] = True
        self.has_red_F[node] = False
        self.has_blue_F[node] = False
    
    def red_player_flag(self, node):
        self.has_red_F[node] = True
    
    def blue_player_flag(self, node):
        self.has_blue_F[node] = True


## Game

In [36]:
import random

class Game:
    def __init__(self, p1, p2, node_amount):
        self.players = [p1, p2]
        self.nodes = []
        for i in range(0, node_amount):
            self.nodes.append(GameNode(i))

    def launch(self):
        # ===== CHOOSE RANDOM PLAYER =====
        current_player = random.choice(self.players)
        displayer = AncestorGameDisplayer(len(self.nodes))
        displayer.show()
        
        keep_going = True

        # ===== GAMEPLAY LOOP =====
        while keep_going:
            # ===== CHECK WIN CONDITION =====
            for p in self.players:
                if p.zones_captured >= len(self.nodes) / 2:
                    print("Player", p.player_id, "wins!")
                    return

            # ===== PLAYER TURN =====
            print(f"\nPlayer {current_player.player_id}'s turn")
            print("Nodes:", [n.id for n in self.nodes])
            print("Actions: 'p i j' (set parent), 'f i' (flag node), 'c i j' (capture), 'c i i' (capture)")

            action = input("> ").strip().split()

            try:
                if action[0] == "p":
                    i, j = self.nodes[int(action[1])], self.nodes[int(action[2])]
                    if current_player.can_give_parent(i, j):
                        i.set_parent(j)
                        displayer.add_edge(i.id, j.id)
                        
                        print(f"Node {i.id}'s parent is now {j.id}")
                        current_player = self.players[0] if current_player == self.players[1] else self.players[1]
                    else:
                        print(f"Node {i.id} has already a parent or adding this edge makes a cycle")

                elif action[0] == "f":
                    i = self.nodes[int(action[1])]
                    if current_player.can_place_flag(i):
                        i.add_flag(Flag(current_player.player_id))
                        if current_player is self.players[0]:
                            displayer.red_player_flag(i.id)
                        else:
                            displayer.blue_player_flag(i.id)
                            
                        print(f"Flag added to node {i}")
                        current_player = self.players[0] if current_player == self.players[1] else self.players[1]
                    else:
                        print(f"Node {i.id} is captured or already has this player's flag")

                elif action[0] == "c":
                    i, j = self.nodes[int(action[1])], self.nodes[int(action[2])]
                    if current_player.can_capture(i, j):
                        current_player.capture(i, j)
                        if current_player is self.players[0]:
                            displayer.red_player_capture(i.id)
                        else:
                            displayer.blue_player_capture(i.id)
                            
                        print(f"Captured nodes between {i.id} and {j.id}!")
                        current_player = self.players[0] if current_player == self.players[1] else self.players[1]
                    else:
                        print(f"Nodes {i.id} and {j.id} don't have this player's flag or do not share an ancestor")

                elif action[0] == "s":
                    keep_going = False

                else:
                    print("Unknown action, try again.")
                    continue

            except (IndexError, ValueError):
                print("Invalid input, try again.")
                continue

            displayer.show()


In [ ]:
game = Game(Player(0), Player(1), 8)
game.launch()